## 一、概念

### 记忆与短期记忆

记忆系统能够记住以往交互的信息。对于人工智能体而言，它使它们能够记住之前的交互，从反馈中学习，并适应用户偏好。

#### 短期记忆

短期记忆功能使应用程序能够记住【单个线程或对话】中的先前交互。

> **进阶点：** 在 AI 领域，通常把 `thread_id` 覆盖的范围称为 **"Session Memory"（会话记忆）**。
>
> **长期记忆 (Long-term)：** 通常依靠 **Vector DB (向量数据库)** 或 **Store** 实现，解决“你三个月前喜欢什么”的问题。

##### LangChain 短期记忆实现核心思路

**Thread ID (线程 ID) 是钥匙：** 数据库就像一个巨大的柜子，`thread_id` 就是柜门上的编号。只要 ID 不变，AI 就能打开柜子取出之前的聊天历史。

**Checkpoint (检查点) 是照片：** 每次对话完，Saver 都会给当前的“对话状态”拍一张快照存进数据库。

**无状态转换：** 实际上模型（Qwen/Gemini）本身永远是无状态的。是 LangChain 在你发新消息前，偷偷去数据库里把老消息翻出来，拼接在一起发给模型。

> **补充：** 在 LangChain 的最新设计（LangGraph）中，它不仅仅是拼接字符串，而是**恢复状态（State）**。
>
> **为什么要区分？** 因为状态里不仅有对话历史（Messages），还可能有你自定义的变量（比如：`current_investigation_step: 3`）。`Saver` 恢复的是整个“运行现场”，而不仅仅是聊天记录。



### 二、用法

#### 线程级记忆

存储在图的状态中，即内存。Agent 可以访问给定对话的完整上下文，同时保持不同线程之间的分离。

##### 示例：自定义代理内存


> ### InjectedState 作用
>
> ####  如果不用 `InjectedState`，会发生什么？
>
> 假设你想让 AI 查用户信息，如果你用传统的方式定义工具：
>
> ```Python
> @tool
> def get_user_info(user_id: str) -> str:
>     # 逻辑代码...
> ```
>
> **这种情况下：**
>
> 1. **模型必须显式知道 `user_id`：** 模型在思考时，必须从对话历史里死记硬背出 `user_123`。
>
> 2. **传输风险：** 模型必须把 `user_id` 作为参数写进 `tool_calls` 里。如果模型幻觉（写错成 `user_456`），或者用户在对话里故意诱导（“忘掉之前的 ID，用 999 查”），工具就会报错或查错。
>
> 3. **Prompt 负担：** 你的提示词里必须不断提醒模型：“别忘了 ID 是 xxx，调工具时记得带上”。
>
>    
>
> #### 用了`InjectedState`
>
> 当你写 `state: Annotated[dict, InjectedState]` 时，你在告诉 LangChain：
>
> > **这个参数不是给模型填的，而是从 Agent 的‘脑回路’里直接截取的。**
> >
> > ##### 它的运行流程是这样的：
> >
> > 1. **模型发出调用：** 模型看到 `get_user_info`，它发现这个工具在它的视角里**没有参数**（或者只有它关心的参数）。它只需要大喊一声：“我要查用户信息！”
> > 2. **LangChain 拦截：** 在工具真正执行前，LangChain 发现这里有一个 `InjectedState` 标记。
> > 3. **自动填充：** LangChain 自动把当前 Agent 的整个 `State`（包括你存的 `user_id` 和 `preferences`）塞进这个 `state` 变量里。
> > 4. **工具执行：** 工具拿到了最准确、由系统维护的 `user_id`，而不是由模型（可能出错）传进来的 `user_id`。




In [ ]:
from typing import Annotated, Any, cast

from langchain.agents import AgentState, create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import InjectedState, tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


class CustomAgentState(AgentState):
    user_id: str
    preferences: dict


@tool
def get_user_info(state: Annotated[dict, InjectedState]) -> str:
    """Look up user info from the current agent state."""
    uid = state.get("user_id", "Unknown")
    prefs = state.get("preferences", {})
    theme = prefs.get("theme", "light")

    return f"查找到用户 {uid}。用户偏好主题：{theme}。该用户目前正在与 Webster 聊天。"


model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

agent = create_agent(
    model=model,
    tools=[get_user_info],
    state_schema=CustomAgentState,
    checkpointer=InMemorySaver(),
)
# invoke 时的输入
input_data: CustomAgentState = {
    "messages": [HumanMessage(content="Hello, I'm Webster")],
    "user_id": "user_123",
    "preferences": {"theme": "dark"},
}
# 坚持要保留 CustomAgentState 类的实例化
# 使用 cast 强转，解决Pylance飘红
# 坑：即使 HumanMessage 是 AnyMessage 的子类，但在 Python 的 list 泛型中，它是不协变的。Pylance 认为你传的是一个纯 HumanMessage 列表，而它需要一个更通用的 AnyMessage | dict 列表。
result = agent.invoke(cast(Any, input_data), {"configurable": {"thread_id": "1"}})


for msg in result["messages"]:
    print(f"[{msg.type}]: {msg.content}")

[human]: Hello, I'm Webster
[ai]: 
[tool]: 查找到用户 user_123。用户偏好主题：dark。该用户目前正在与 Webster 聊天。
[ai]: Hello, Webster! It seems you prefer a dark theme. How can I assist you today?


* 第二次提问，不携带 user_id，验证短期记忆是否生效。

In [ ]:
input_data_2 = {
    "messages": [HumanMessage(content="My ID was what? and what is my theme?")]
}

# 注意：thread_id 必须保持一致，依然是 "1"
result_2 = agent.invoke(cast(Any, input_data_2), {"configurable": {"thread_id": "1"}})

# 打印结果看看
for msg in result_2["messages"]:
    print(f"[{msg.type}]: {msg.content}")

[human]: Hello, I'm Webster
[ai]: 
[tool]: 查找到用户 user_123。用户偏好主题：dark。该用户目前正在与 Webster 聊天。
[ai]: Hello, Webster! It seems you prefer a dark theme. How can I assist you today?
[human]: My ID was what? and what is my theme?
[ai]: Your user ID is `user_123`, and it appears that your preferred theme is dark. Is there anything specific you need help with or any changes you'd like to make?


#### 线程记忆持久化

使用由数据库支持的检查点

##### SqliteSaver or PostgresSaver 

`PostgresSaver` 就像是一个**翻译官**。它知道怎么把 LangChain 的聊天记录（Messages）翻译成 SQL 语句并存入表中。但它工作的前提是：**得有一个正在运行的数据库等它去存。**

如果你在代码里用了 `PostgresSaver`，你必须在本地（或者云端）安装并启动一个 **PostgreSQL** 数据库。

- 官方文档之所以推荐 `PostgresSaver`，是因为在真正的服务器上，PG 是最稳健、最通用的选择。

- demo 演示最好用轻量的 SqliteSaver ，因为API都一样，LangChain完成了最大程度的适配，类似于ORM。

  >  严格来说，LangChain 的 `BaseCheckpointSaver` 定义了一套标准接口。无论底层是 SQLite、Postgres 还是 Redis，对于上层 Agent 代码来说，**调用的命令完全一致**。这就是“解耦”的魅力。

##### 示例：SqliteSaver

> ### TypedDict vs AgentState
>
> #### 1. 灵活性：`TypedDict` 是“协议”，而 `AgentState` 是“类”
>
> 在 LangChain 的新版设计（尤其是基于 LangGraph 的 Agent）中，状态被看作是一个**纯粹的数据结构**，而不是一个带有复杂行为的对象。
>
> - **`AgentState` (类继承)：** 当你继承一个类时，你实际上是把一大堆父类的“包袱”都背上了。在进行序列化（也就是把状态存进 SQLite 数据库）时，类实例有时会产生一些难以预料的副作用。
> - **`TypedDict` (类型字典)：** 它本质上就是一个**普通的 Python 字典**，只是加了类型提示。
>   - **容易存：** 数据库存字典非常快，格式转换也简单。
>   - **好理解：** 对于 LLM 框架来说，它只需要知道“键（Key）”和“值（Value）”是什么
>
> #### 2. `operator.add` 的魔力
>
> 你注意到了 `messages: Annotated[list[AnyMessage], operator.add]` 这一行吗？这是使用 `TypedDict` 的**核心理由**。
>
> - **Reducer (还原器) 机制：** 在 LangGraph 中，我们需要定义**当新数据进入状态时，该如何处理旧数据**。
> - **类继承的局限：** 如果直接用类，定义这种“合并逻辑”会变得很臃肿。
> - **TypedDict + Annotated：** 这种组合非常优雅地告诉 LangChain：“每当 Agent 产生新的消息时，**不要覆盖**旧消息，而是使用 `operator.add` 把它们**追加**到列表末尾。”
>
> `TypedDict` 就像是一个**活页本**，你可以通过 `Annotated` 规定每一页是直接替换还是往后粘贴。而传统的 `class` 更像是一本订装好的书，修改起来没那么灵活。

In [19]:

import operator
from typing import TypedDict

from langchain.messages import AnyMessage
# 核心变化：导入 SqliteSaver
from langgraph.checkpoint.sqlite import SqliteSaver

# 1. 定义状态
class CustomAgentStateTypedDict(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    user_id: str
    preferences: dict

# 2. 定义工具(复用)

# 3. 初始化模型(复用)

# 4. 配置持久化存储 (Sqlite)
# 这行代码会在当前目录下创建一个名为 checkpoints.db 的 SQLite 数据库文件
memory_context = SqliteSaver.from_conn_string("checkpoints.db")

with memory_context as memory:
    # 5. 创建Agent
    agent = create_agent(
        model=model,
        tools=[get_user_info],
        state_schema=CustomAgentStateTypedDict,
        checkpointer=memory,
    )
    config = {"configurable": {"thread_id": "webster_session_001"}}

    input_data = {
        "messages": [HumanMessage(content="Hello, I'm Webster")],
        "user_id": "user_123",
        "preferences": {"theme": "dark"},
    }

    print("--- 第一次运行 ---")
    result = agent.invoke(cast(Any, input_data), config)
    print(f"AI 回复: {result['messages'][-1].content}")


--- 第一次运行 ---
AI 回复: PålÅ It's nice to meet you again, Webster! How can I assist you today? If you're looking for your ID, it appears to be user_123. Do you have any specific questions or need help with something in particular?


* 第二次运行，取消传递user_id，验证存储功能是否生效？

In [18]:
memory_context = SqliteSaver.from_conn_string("checkpoints.db")
with memory_context as memory:
    # 重新创建 agent（或者确保它使用的是这个 memory 实例）
    agent = create_agent(
        model=model,
        tools=[get_user_info],
        state_schema=CustomAgentStateTypedDict,
        checkpointer=memory,
    )
    
    config = {"configurable": {"thread_id": "webster_session_001"}}

    # 第二次运行：注意这里完全没有传 user_id
    input_data_2 = {
        "messages": [HumanMessage(content="Hello, What's my ID?")],
    }

    print("--- 第二次运行 ---")
    result = agent.invoke(cast(Any, input_data_2), config)
    print(f"AI 回复: {result['messages'][-1].content}")

--- 第二次运行 ---
AI 回复: Your ID is user_123, Webster. If you have any other questions or need further assistance, feel free to ask!



> ### with 语句操作manager
>
> #### 1. 核心原因：门已经关了，钥匙失效了
>
> 当你执行完第一个 `with memory_context as memory:` 块时，Python 会自动触发一个动作：**关闭数据库连接并清理资源**。
>
> - **第一次运行：** 你进入 `with`，门开了，`memory` 是有效的，Agent 成功把数据存进了 `checkpoints.db`。
> - **出 `with` 块：** 只要代码缩进回到了左侧，Python 就认为你“用完了”，它会**立刻锁上门**。
> - **第二次运行：** 此时如果你还在尝试使用那个已经失效的 `memory` 对象（或者相关的上下文），但连接已经断开，所以报错了。
>
> #### 2. 为什么你“必须重新写一遍”或者“合在一起写”？
>
> 这就是为什么你要么得重新 `from_conn_string`（重新拿一把钥匙开一次门），要么就得把两次运行**都塞进同一个 `with` 缩进里**。
>
> ```python
> # 第一次开门，干完活，关门
> with SqliteSaver.from_conn_string("checkpoints.db") as memory:
>     agent.invoke(input_data, config)
> 
> # 此时 memory 已经死了！
> 
> # 第二次必须重新申请钥匙开门
> with SqliteSaver.from_conn_string("checkpoints.db") as memory:
>     # 哪怕再次创建 agent 也没关系，它会去读那个 .db 文件
>     agent = create_agent(..., checkpointer=memory) 
>     agent.invoke(input_data_2, config)
> ```
>
> #### 3. 为什么你会看到 `_GeneratorContextManager` 报错？
>
> 当你写 `memory_context = SqliteSaver.from_conn_string(...)` 时：
>
> - `memory_context` 本身只是一个 **“协议/计划书”**，它不是真正的存储对象。
> - 只有当你执行 `with memory_context as memory` 时，Python 才会执行这个计划，把真正的存储对象赋值给 `memory`。
>
> 你在报错的代码里，可能是在 `with` 块外面直接使用了 `memory_context`，或者尝试重复利用已经关闭的上下文，导致它抛出了一个关于“生成器管理器（GeneratorContextManager）”的内部错误。